# A3: NDVI by class, EuroSAT against Thailand

NDVI is one number per pixel that says how green the ground is:

`NDVI = (B8 - B4) / (B8 + B4)`   (near infrared minus red, divided by their sum)

Dense green vegetation reflects a lot of near infrared and absorbs red, so NDVI is high (about 0.6 to 0.9). Bare soil, roofs and roads sit
near zero. Water absorbs near infrared and is negative. NDVI uses only two of the 12 bands, so it shows how green a class is, not everything
that step A2 shows (for example the short-wave infrared, where crop differs a lot).

The pixels are the same as in A2 (same seed, same order of drawing), so the figures of A2 and A3 describe the same pixels:
EuroSAT training patches, and the pure Thai patches in L1C and in L2A. The same number of pixels is drawn from forest, crop, built-up and water.
Grassland is left out because no Thai patch passes the purity threshold.

**Keep in mind**
- The Thai images are a January-February median composite (dry season); every EuroSAT patch is a single date from any season.
- Each Thai class comes from one or two study areas (see `DATA_NOTES.md`).
- Section 5 is a small illustration with NDVI alone. It is not the model of the project.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import config as c

# chart colours on a light surface: categorical slots 1-3 of the data-viz palette (same as A2)
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
COLORS = {"EuroSAT (train)": "#2a78d6", "Thai L1C": "#eb6834", "Thai L2A": "#1baf7a"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.size": 10,
})

OUT = c.OUTPUT_DIR / "a3"
OUT.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    # Windows sometimes refuses to open a file that was written a moment ago, so retry a few times
    path = OUT / name
    for attempt in range(5):
        try:
            fig.savefig(path, dpi=130, bbox_inches="tight")
            break
        except OSError:
            if attempt == 4:
                raise
            time.sleep(1)
    print("saved", path)

## 1. Load the data and draw the pixels

Exactly the same procedure as A2: up to 300 random patches per class, then 100,000 random pixels out of them, seed 42. The two Thai products use the
same patches and pixels. For the rule in section 5 a second set of EuroSAT pixels is drawn from the **validation** patches, so the rule is not tested on the pixels it was set from.

In [ ]:
EU_X = np.load(c.EUROSAT_DIR / "eurosat_X.npy", mmap_mode="r")      # (27000, 12, 64, 64)
EU_Y = np.load(c.EUROSAT_DIR / "eurosat_y.npy")
EU_SPLIT = np.load(c.EUROSAT_DIR / "eurosat_split.npy")              # 0 train, 1 val, 2 test
TH_X = {"Thai L1C": np.load(c.PATCH_DIR / "thailand_X_L1C.npy", mmap_mode="r"),
        "Thai L2A": np.load(c.PATCH_DIR / "thailand_X_L2A.npy", mmap_mode="r")}
TH = pd.read_csv(c.PATCH_DIR / "thailand_patches.csv", keep_default_na=False)   # "nan" is an area name here

EVAL = c.THAI_EVAL_CLASSES      # forest, crop, built-up, water
PATCHES_PER_CLASS = 300
PIXELS_PER_CLASS = 100_000


def draw(patch_ids, rng):
    ids = np.sort(rng.choice(patch_ids, size=min(PATCHES_PER_CLASS, len(patch_ids)), replace=False))
    pix = np.sort(rng.choice(len(ids) * 64 * 64, size=min(PIXELS_PER_CLASS, len(ids) * 64 * 64), replace=False))
    return ids, pix


def take(X, ids, pix):
    block = np.asarray(X[ids])                                  # (patches, 12, 64, 64)
    return block.transpose(0, 2, 3, 1).reshape(-1, 12)[pix].astype("float32")


def ndvi(pixels):
    b8, b4 = pixels[:, c.BANDS.index("B8")], pixels[:, c.BANDS.index("B4")]
    total = b8 + b4
    values = np.where(total > 0, (b8 - b4) / np.where(total > 0, total, 1), np.nan)
    return values[np.isfinite(values)]


rng = np.random.default_rng(c.SEED)              # same order of drawing as in A2
pixels = {name: {} for name in COLORS}
for cls in EVAL:
    eu_ids = np.flatnonzero((EU_Y == c.CLASSES.index(cls)) & (EU_SPLIT == 0))
    ids, pix = draw(eu_ids, rng)
    pixels["EuroSAT (train)"][cls] = take(EU_X, ids, pix)
    th_ids = TH.loc[TH["pure"] & (TH["label"] == cls), "patch_id"].to_numpy()
    ids, pix = draw(th_ids, rng)
    for name, X in TH_X.items():
        pixels[name][cls] = take(X, ids, pix)

NDVI = {name: {cls: ndvi(px) for cls, px in per_class.items()} for name, per_class in pixels.items()}
NDVI_MIX = {name: np.concatenate([NDVI[name][cls] for cls in EVAL]) for name in COLORS}
print({name: len(v) for name, v in NDVI_MIX.items()})

## 2. Histograms of NDVI, one panel per class

A curve further to the right means greener ground. The vertical axis is the share of pixels per value (density), so its numbers carry no meaning and are hidden.
The last panel mixes the four classes in equal parts.

In [ ]:
def ndvi_panel(ax, data, title):
    bins = np.linspace(-0.6, 1.0, 81)
    for name, values in data.items():
        ax.hist(np.clip(values, -0.6, 1.0), bins=bins, density=True, histtype="step", linewidth=2, color=COLORS[name])
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_yticks([])
    ax.set_xlim(-0.6, 1.0)


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, cls in zip(axes.ravel(), EVAL):
    ndvi_panel(ax, {name: NDVI[name][cls] for name in COLORS}, cls)
ndvi_panel(axes[1, 1], NDVI_MIX, "all four classes in equal parts")
axes[1, 2].axis("off")
for ax in axes[1, :2]:
    ax.set_xlabel("NDVI")
handles = [Line2D([0], [0], color=col, linewidth=2.5, label=name) for name, col in COLORS.items()]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("NDVI of the same class in Europe and in Thailand", y=1.04, fontsize=13, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, "ndvi_histograms.png")
plt.show()

## 3. The numbers behind the figure

Median, mean and the 5th, 25th, 75th and 95th percentile of NDVI for every class and source. Saved as `outputs/a3/ndvi_stats.csv`.

In [ ]:
records = []
groups = {**{cls: {name: NDVI[name][cls] for name in COLORS} for cls in EVAL}, "balanced mix": NDVI_MIX}
for group, data in groups.items():
    for name, v in data.items():
        records.append({"group": group, "source": name, "mean": v.mean(), "median": np.median(v),
                        "p05": np.percentile(v, 5), "p25": np.percentile(v, 25), "p75": np.percentile(v, 75),
                        "p95": np.percentile(v, 95)})
stats = pd.DataFrame(records)
stats.round(3).to_csv(OUT / "ndvi_stats.csv", index=False)

median = stats.pivot(index="group", columns="source", values="median").reindex(EVAL + ["balanced mix"])
median["Thai L1C - EuroSAT"] = median["Thai L1C"] - median["EuroSAT (train)"]
median["Thai L2A - EuroSAT"] = median["Thai L2A"] - median["EuroSAT (train)"]
median.round(2)

## 4. How far apart are the classes inside one region?

If the gap between two classes in NDVI is small, a model that relies on NDVI confuses them easily. The median NDVI of every class, EuroSAT beside Thai L1C:

In [ ]:
order = stats[stats["group"].isin(EVAL)].pivot(index="group", columns="source", values="median").reindex(EVAL)
order[["EuroSAT (train)", "Thai L1C", "Thai L2A"]].round(2).sort_values("EuroSAT (train)", ascending=False)

## 5. An illustration: what if only NDVI were known?

A rule with no learning: take the median NDVI of every class in the EuroSAT training pixels, and give a pixel the class whose median is closest.
Then apply it to pixels it has not seen: EuroSAT validation, Thai L1C and Thai L2A. The share of pixels that get their own class back, per class:

This is a single index and a single pixel at a time, far simpler than the models of the project. It only shows how much a shift in NDVI alone can cost.

In [ ]:
medians = np.array([np.median(NDVI["EuroSAT (train)"][cls]) for cls in EVAL])


def predict(values):
    return np.argmin(np.abs(values[:, None] - medians[None, :]), axis=1)


rng_val = np.random.default_rng(c.SEED + 1)
val = {}
for cls in EVAL:
    ids, pix = draw(np.flatnonzero((EU_Y == c.CLASSES.index(cls)) & (EU_SPLIT == 1)), rng_val)
    val[cls] = ndvi(take(EU_X, ids, pix))

tests = {"EuroSAT validation": val, "Thai L1C": NDVI["Thai L1C"], "Thai L2A": NDVI["Thai L2A"]}
rows = {}
for name, data in tests.items():
    rows[name] = {cls: (predict(data[cls]) == k).mean() for k, cls in enumerate(EVAL)}
    rows[name]["mean of the four"] = np.mean(list(rows[name].values()))
recall = pd.DataFrame(rows).T
recall.round(2).to_csv(OUT / "ndvi_rule_recall.csv")
(recall * 100).round(0).astype(int).astype(str) + "%"

Where do the Thai L1C pixels go? Rows are the true class, columns the class the rule gives (share of the row):

In [ ]:
confusion = pd.DataFrame([np.bincount(predict(NDVI["Thai L1C"][cls]), minlength=len(EVAL)) / len(NDVI["Thai L1C"][cls]) for cls in EVAL],
                         index=EVAL, columns=EVAL)
(confusion * 100).round(0).astype(int).astype(str) + "%"

## 6. Questions to answer for the slide

- Which classes are greener or less green in Thailand than in EuroSAT, and by how much (section 3)?
- Do the Thai curves overlap with the EuroSAT curves of the same class, or with those of another class (sections 2 and 5)?
- Which classes are close to each other in NDVI inside one region (section 4)? Those are the ones a model confuses first.
- Which part of the difference can come from the dry-season composite?